# Custom Dataset YOLOv11 Fine-tuning & ONNX Export
본 노트북은 n개의 클래스로 구성된 640x640 크기의 이미지를 학습하고, 완료 후 ONNX 모델로 내보내는 파이프라인을 포함하고 있습니다.

## 1. 구글 드라이브 마운트

In [1]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('g-drive mounted.')
    colab = True
except:
    print('local drive.')
    colab = False

Mounted at /content/drive
g-drive mounted.


## 2. 경로 설정 및 데이터셋 준비
환경에 맞게 `data.yaml`의 경로를 수정해 주세요. 본 예시는 기존 노트북 구조를 유지하여 작성되었습니다.

In [2]:
import os

if colab:
    # 코랩 환경에서의 경로 예시
    DATA_YAML = '/content/drive/MyDrive/Classroom/Project_TEAM6/data.yaml'
    project_root = '/content/drive/MyDrive/Classroom/Project_TEAM6'
else:
    # 로컬 환경에서의 경로 예시
    DATA_YAML = './dataset/data.yaml'
    project_root = './custom_runs'

# unzip
!unzip -q /content/drive/MyDrive/Classroom/Project_TEAM6/Dataset.zip -d /content

print(f"사용 예정 data.yaml 경로: {DATA_YAML}")

사용 예정 data.yaml 경로: /content/drive/MyDrive/Classroom/Project_TEAM6/data.yaml


### 2-1. xml to txt convert

In [3]:
# 클래스 이름 순서 (data.yaml과 반드시 일치해야 함)
# CLASSES = ['크라운쵸코하임284G', '농심벌집핏자90G', '농심포스틱84G', '빙그레꽃게랑오리지널맛70G', '꽃게랑불짬뽕맛70G', '꽃게랑와사비70G' ]

import csv

csv_file_path = '/content/drive/MyDrive/Classroom/Project_TEAM6/class.csv'
CLASSES = []
with open(csv_file_path, mode='r', encoding='cp949') as f:
  reader = csv.reader(f)

  next(reader, None)

  for row in reader:
    if len(row) > 1:
      CLASSES.append(row[1].strip())

In [4]:
CLASSES

['농심벌집핏자90G', '농심포스틱84G', '빙그레꽃게랑오리지널맛70G', '크라운)꽃게랑불짬뽕맛70G', '크라운)꽃게랑와사비70G']

In [5]:
import xml.etree.ElementTree as ET
import os

dataset_root = "/content/Dataset"

def convert(size, box):
    dw = 1. / size[0]
    dh = 1. / size[1]
    x = (box[0] + box[2]) / 2.0
    y = (box[1] + box[3]) / 2.0
    w = box[2] - box[0]
    h = box[3] - box[1]
    return (x * dw, y * dh, w * dw, h * dh)

def convert_folder(src_folder, dst_labels_folder):
    # 하위 폴더 경로 구조 그대로 생성 (예: labels/30060)
    if not os.path.exists(dst_labels_folder):
        os.makedirs(dst_labels_folder)

    xml_files = [f for f in os.listdir(src_folder) if f.endswith('.xml')]
    if not xml_files:
        return

    for filename in xml_files:
        in_file_path = os.path.join(src_folder, filename)
        out_file_path = os.path.join(dst_labels_folder, filename.replace('.xml', '.txt'))

        tree = ET.parse(in_file_path)
        root = tree.getroot()

        size = root.find('size')
        if size is None: continue
        w = int(size.find('width').text)
        h = int(size.find('height').text)

        with open(out_file_path, 'w', encoding='utf-8') as out_file:
            for obj in root.iter('object'):
                cls = obj.find('name').text
                if cls not in CLASSES:
                    continue
                cls_id = CLASSES.index(cls)

                xmlbox = obj.find('bndbox')
                b = (float(xmlbox.find('xmin').text),
                     float(xmlbox.find('ymin').text),
                     float(xmlbox.find('xmax').text),
                     float(xmlbox.find('ymax').text))

                bb = convert((w, h), b)
                out_file.write(f"{cls_id} {' '.join([f'{a:.6f}' for a in bb])}\n")

# === 하위 디렉토리 구조를 그대로 유지하며 변환 ===
sub_modes = ["Training", "Validation"]

for mode in sub_modes:
    base_label_dir = os.path.join(dataset_root, mode, "label")

    if os.path.exists(base_label_dir):
        sub_dirs = [d for d in os.listdir(base_label_dir) if os.path.isdir(os.path.join(base_label_dir, d))]

        for sub_dir in sub_dirs:
            src_folder = os.path.join(base_label_dir, sub_dir)
            # 💡 dst 경로에 sub_dir을 유지하여 'labels/30060' 형태로 저장되도록 함
            dst_labels_folder = os.path.join(dataset_root, mode, "labels", sub_dir)

            convert_folder(src_folder, dst_labels_folder)
        print(f"변환 완료: {mode} 완료")

변환 완료: Training 완료
변환 완료: Validation 완료


## 3. Ultralytics 패키지 설치

In [6]:
!pip -q install -U ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 29.8 MB/s eta 0:00:00


## 4. 모델 로드 및 학습 진행 (640*640 설정)
n개의 클래스를 인식하는 `data.yaml` 정보와 함께 이미지 크기를 **640**으로 설정하여 학습을 진행합니다.

In [7]:
from ultralytics import YOLO

# 최신 프리트레인드 YOLO11 가벼운 모델 로드
model = YOLO('yolo11n.pt')

# 학습 시작
results = model.train(
    data=DATA_YAML,
    epochs=50,          # 학습 에포크 (원하는 만큼 수정 가능)
    batch=16,           # 배치 사이즈
    imgsz=640,          # 요청 조건: 640*640 크기 설정
    device=0,           # GPU 번호 (CUDA:0)
    workers=4,
    project=project_root,
    name='Project_TEAM6_trained'
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.75 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Classroom/Project_TEAM6/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015,

## 5. 최적 성능 가중치 파일 로드 및 ONNX 변환(Export)

In [8]:
# 학습 완료 후 검증 성능이 가장 좋았던 best.pt 경로 확보
best_model_path = os.path.join(model.trainer.save_dir, 'weights', 'best.pt')
best_model = YOLO(best_model_path)

print(f"최적 가중치 모델 로드 완료: {best_model_path}")

# ONNX 파일 형식으로 익스포트
# imgsz=640으로 지정하여 640x640 정적 입력 해상도를 고정할 수 있습니다.
onnx_path = best_model.export(format='onnx', imgsz=640)

print(f"ONNX 파일 내보내기 성공: {onnx_path}")

최적 가중치 모델 로드 완료: /content/drive/MyDrive/Classroom/Project_TEAM6/Project_TEAM6_trained-2/weights/best.pt
Ultralytics 8.4.75 🚀 Python-3.12.13 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLO11n summary (fused): 101 layers, 2,583,127 parameters, 0 gradients, 6.3 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/Classroom/Project_TEAM6/Project_TEAM6_trained-2/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 9, 8400) (5.2 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 12 packages in 420ms
Prepared 4 packages in 3.94s
Installed 4 packages in 600ms
 + colorama==0.4.6
 + onnx==1.22.0
 + onnxruntime==1.27.0
 + onnxslim==0.1.94

requirements: AutoUpdate success ✅ 5.8s
WARNING